In [1]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from datasets import load_dataset
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import defaultdict
import re

In [2]:
df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/qa_test.csv')

In [3]:
df.head()

,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,5cf39741-d5bd-4226-a94d-ff0c13ca5eaa,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,41196666-5ad7-4651-98aa-9fa9ddb4aad5,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,6b6a74ae-4b78-4a17-a338-5418ac1408cb,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,e94de938-6049-4e4f-9ac1-b6f98884e235,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,2943f760-b273-4602-bd66-6e5c73ae0edf,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [4]:
data=df[['question','long_answers']]

In [5]:
# read evidence data
evidence_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/evidence_train.csv')
evidence_ls = evidence_df.loc[:,'text'].dropna().to_list()

In [6]:
# load embedding model
model_path = '/raid/deallab/SF_RAG_Data/ASQA/models/fine_tuned_model_64'
model = SentenceTransformer(model_path)

# load model to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# calculate embeddings for all data
evidence_embeddings = model.encode(evidence_ls, convert_to_tensor=True).to(device)

In [7]:
questions=data['question']

In [8]:
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=quantization_config,
    device_map="auto"
)

tokenizer_gen.pad_token = tokenizer_gen.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gen.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [9]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)

    query_embedding = query_embedding.unsqueeze(0)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10]
    print(top_results)
    res={}
    for idx in top_results:
        tmp=evidence_ls[idx]
        res[tmp]=similarities[idx]
        
    return list(res.keys())

In [10]:
references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]

In [11]:
references[3]

{'id': 'e94de938-6049-4e4f-9ac1-b6f98884e235',
 'sample_id': -881464876144297194,
 'question': 'Who played the weasley brothers in harry potter?',
 'follow_up_questions': "['Who played  Bill weasley in Harry Potter and the Prisoner of Azkaban?', 'Who played percy weasley in harry potter?', 'Who played fred weasley in harry potter?', 'Who played ron weasley in harry potter?', 'Who played george weasley in harry potter?', 'Who played  Bill weasley in harry potter (2001-2011)?']",
 'long_answers': '[\'Rupert Grint played Ron Weasley in all the Harry Potter films.  Richard Fish appeared as Bill Weasley briefly in the film adaptation of "Harry Potter and the Prisoner of Azkaban". Domhnall Gleeson plays Bill Weasley in "Harry Potter and the Deathly Hallows" and the roller coaster ride "Harry Potter and the Escape from Gringotts" at The Wizarding World of Harry Potter – Diagon Alley in Universal Studios Florida. James Phelps and Oliver Phelps played Fred and George Weasley in the Harry Potter

In [12]:
len(questions)

944

In [18]:
from evaluation import evaluate

scores_list=[]
for i in range(len(questions)):
    print(f"Query {i+1} : {questions[i]}")
    print("-"*100)
    results = retrieve_documents(questions[i])
    
    print("Retrieved doc :")
    for j in range(len(results)):
        print(f"\tRank {j} : {results[j]}")
        
    prompt = f"""
    Context information is below.
    ---------------------
    {results}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {questions[i]}
    Answer:
    """
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    candidate = [re.sub('\n|<\|eot_id\|>', '', res)]
    print(candidate)
    print(references[i])

    scores=evaluate(candidate,[references[i]])
    print(scores)
    scores_list.append(scores)

Query 1 : Who has the highest goals in world football?
----------------------------------------------------------------------------------------------------
tensor([141763,  48264,  48248,  48265, 141709,  15415,  15401,  48246,  48247,
         48249], device='cuda:0')
Retrieved doc :
	Rank 0 : ] * rsssf record for most seasons with over 100 top level goals scored ( including friendlies ) : 3 ( 1959, 1961, 1965 ) [ 373 ] * rsssf record for most goals scored before the age of 30 : 675 [ 374 ] * rsssf record for most top level career goals ( including friendlies ) : 1, 274 [ 375 ] * guinness world record for most career goals in world football ( including friendlies ) : 1, 283 ( in 1, 363 games ) [ 376 ] * iffhs record for most top division league goals : 604 [ 362 ] [ 377 ] * iffhs record for most top level domestic goals : 659 [ 362 ] [ 377 ] * guinness world record for most hat - tricks in world football : 92 [ 378 ] [ 379 ] * most hat - tricks for brazil : 7 [ 380 ] * most fifa world

In [19]:
scores_df=pd.DataFrame(scores_list)

In [20]:
scores_df.mean()

rougeLsum    28.315600
length       64.359110
str_em       19.200212
ovscore      14.931574
dtype: float64

In [21]:
scores_df.to_csv('/raid/deallab/SF_RAG_Data/ASQA/baseline_results.csv', index=False)